In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from churn_prediction.paths import  INTERIM_NOTEBOOK_ONLINE_RETAIL_DIR,PROCESSED_ONLINE_RETAIL_DIR

All directories created successfully!


In [3]:
df = pd.read_parquet(INTERIM_NOTEBOOK_ONLINE_RETAIL_DIR / 'online_retail_raw.parquet')
print(f"Raw shape: {df.shape}")
print(f"Unique customers (raw): {df['Customer ID'].nunique()}")

Raw shape: (1067371, 8)
Unique customers (raw): 5943


### Chuẩn Hóa dữ liệu

In [4]:
for col in ['Invoice', 'StockCode', 'Description', 'Country']:
    df[col] = df[col].astype(str)

df['Customer ID'] = pd.to_numeric(df['Customer ID'], errors='coerce')
df['Quantity']    = pd.to_numeric(df['Quantity'],    errors='coerce')
df['Price']       = pd.to_numeric(df['Price'],       errors='coerce')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

### Kiểm tra trước khi cleaning

In [5]:
print(f"Total rows  : {len(df):>10,}")
print(f"Cancelled invoices (C*) : {df['Invoice'].str.startswith('C').sum():>10,}")
print(f"Quantity <= 0   : {(df['Quantity'] <= 0).sum():>10,}")
print(f"Price <= 0  : {(df['Price'] <= 0).sum():>10,}")
print(f"Missing Customer ID : {df['Customer ID'].isna().sum():>10,}")
print(f"Duplicate rows  : {df.duplicated().sum():>10,}")

Total rows  :  1,067,371
Cancelled invoices (C*) :     19,494
Quantity <= 0   :     22,950
Price <= 0  :      6,207
Missing Customer ID :    243,007
Duplicate rows  :     34,335


### Cleaning

In [6]:
df_clean = df.copy()

# 1. Bỏ hóa đơn hủy
df_clean = df_clean[~df_clean['Invoice'].str.startswith('C')]
print(f"After removing cancelled : {len(df_clean):>10,}")

# 2. Bỏ Quantity <= 0
df_clean = df_clean[df_clean['Quantity'] > 0]
print(f"After Quantity > 0       : {len(df_clean):>10,}")

# 3. Bỏ Price <= 0
df_clean = df_clean[df_clean['Price'] > 0]
print(f"After Price > 0          : {len(df_clean):>10,}")

# 4. Bỏ missing Customer ID (không thể làm feature engineering)
df_clean = df_clean[df_clean['Customer ID'].notna()]
print(f"After drop NaN CustomerID: {len(df_clean):>10,}")

# 5. Bỏ duplicate — đặt CUỐI CÙNG sau khi đã filter
df_clean = df_clean.drop_duplicates()
print(f"After drop duplicates    : {len(df_clean):>10,}")

print(f"\nUnique customers (clean) : {df_clean['Customer ID'].nunique():>10,}")

After removing cancelled :  1,047,877
After Quantity > 0       :  1,044,420
After Price > 0          :  1,041,670
After drop NaN CustomerID:    805,549
After drop duplicates    :    779,425

Unique customers (clean) :      5,878


### Lưu

In [ ]:
missing = df_clean.isnull().sum()
print("Missing values in clean data:")
print(missing[missing > 0] if missing.any() else "None")

# Lưu đúng df_clean, không phải df
df_clean.to_parquet(
    PROCESSED_ONLINE_RETAIL_DIR / 'online_retail_clean.parquet',
    index=False
)
print(f"\nSaved: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} cols")

Missing values in clean data:
None

Saved: 779,425 rows, 8 cols
